# Crop Recommendation System — ML Pipeline
### M32895 Big Data Applications Coursework

In [2]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("Crop_recommendation.csv")

print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Column names: {list(df.columns)}")
df.head(10)

Dataset shape: 2200 rows x 8 columns
Column names: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice
5,69,37,42,23.058049,83.370118,7.073454,251.055000,rice
6,69,55,38,22.708838,82.639414,5.700806,271.324860,rice
7,94,53,40,20.277744,82.894086,5.718627,241.974195,rice
8,89,54,38,24.515881,83.535216,6.685346,230.446236,rice
9,68,58,38,23.223974,83.033227,6.336254,221.209196,rice


## 2. Data Validation

In [5]:
print("=== Data Types ===")
print(df.dtypes)
print(f"\nAll feature columns are numeric: {all(df.drop('label', axis=1).dtypes.apply(lambda x: np.issubdtype(x, np.number)))}")

=== Data Types ===
N                int64
P                int64
K                int64
temperature    float64
humidity       float64
ph             float64
rainfall       float64
label              str
dtype: object

All feature columns are numeric: True


In [6]:
print("=== Missing Values ===")
missing = df.isnull().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

=== Missing Values ===
N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0
dtype: int64

Total missing values: 0


In [7]:
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
    print("Removing duplicates...")
    df = df.drop_duplicates()
    print(f"Shape after removing duplicates: {df.shape}")
else:
    print("No duplicates found — dataset is clean.")

Number of duplicate rows: 0
No duplicates found — dataset is clean.


In [8]:
print("=== Statistical Summary ===")
df.describe().round(2)

=== Statistical Summary ===


,N,P,K,temperature,humidity,ph,rainfall
count,2200.00,2200.00,2200.00,2200.00,2200.00,2200.00,2200.00
mean,50.55,53.36,48.15,25.62,71.48,6.47,103.46
std,36.92,32.99,50.65,5.06,22.26,0.77,54.96
min,0.00,5.00,5.00,8.83,14.26,3.50,20.21
25%,21.00,28.00,20.00,22.77,60.26,5.97,64.55
50%,37.00,51.00,32.00,25.60,80.47,6.43,94.87
75%,84.25,68.00,49.00,28.56,89.95,6.92,124.27
max,140.00,145.00,205.00,43.68,99.98,9.94,298.56


In [9]:
print("=== Value Range Validation ===")
for col in df.select_dtypes(include=[np.number]).columns:
    print(f"{col:>12s}: min = {df[col].min():.2f}, max = {df[col].max():.2f}")

ph_valid = df['ph'].between(0, 14).all()
print(f"\npH values within valid range (0-14): {ph_valid}")

nutrients_valid = (df[['N', 'P', 'K']] >= 0).all().all()
print(f"All nutrient values non-negative: {nutrients_valid}")

=== Value Range Validation ===
           N: min = 0.00, max = 140.00
           P: min = 5.00, max = 145.00
           K: min = 5.00, max = 205.00
 temperature: min = 8.83, max = 43.68
    humidity: min = 14.26, max = 99.98
          ph: min = 3.50, max = 9.94
    rainfall: min = 20.21, max = 298.56

pH values within valid range (0-14): True
All nutrient values non-negative: True


## 3. Data Preparation

In [10]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

print("=== Label Mapping ===")
for crop, code in sorted(zip(le.classes_, le.transform(le.classes_)), key=lambda x: x[1]):
    print(f"  {code:2d} -> {crop}")

X = df.drop(['label', 'label_encoded'], axis=1)
y = df['label_encoded']

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print(f"Number of classes: {len(le.classes_)}")

=== Label Mapping ===
   0 -> apple
   1 -> banana
   2 -> blackgram
   3 -> chickpea
   4 -> coconut
   5 -> coffee
   6 -> cotton
   7 -> grapes
   8 -> jute
   9 -> kidneybeans
  10 -> lentil
  11 -> maize
  12 -> mango
  13 -> mothbeans
  14 -> mungbean
  15 -> muskmelon
  16 -> orange
  17 -> papaya
  18 -> pigeonpeas
  19 -> pomegranate
  20 -> rice
  21 -> watermelon

Features shape: (2200, 7)
Target shape:   (2200,)
Number of classes: 22
